# 05 Stress Testing And Risk Metrics

Run deterministic stress scenarios, spot-volatility heatmaps, Monte Carlo P&L, VaR, and Expected Shortfall.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
positions = load_portfolio_config()
stress_table = standard_stress_table(positions, config)
save_table(stress_table, "05_standard_stress_table.csv")
stress_table.head(10)


In [ ]:
matrix = spot_vol_stress_matrix(positions, config["stress"]["spot_shocks"], config["stress"]["volatility_relative_shocks"])
save_table(matrix, "05_spot_vol_stress_matrix.csv")
plot_heatmap(matrix, "Portfolio stress P&L heatmap", "05_portfolio_stress_pnl_heatmap.png")
matrix


In [ ]:
pnl = monte_carlo_portfolio_pnl(positions, n_scenarios=int(config["stress"]["monte_carlo_scenarios"]), horizon_days=float(config["stress"]["monte_carlo_horizon_days"]), seed=int(config["random_seed"]))
risk = var_expected_shortfall(pnl, config["stress"]["var_levels"])
assert risk["ES_95"] >= risk["VaR_95"] and risk["VaR_99"] >= risk["VaR_95"]
print("VALIDATION PASSED: VaR and Expected Shortfall signs and ordering are sensible")
save_output(pnl.to_frame(), "05_monte_carlo_pnl.csv")
save_output(risk, "05_var_expected_shortfall.json")
plt.figure()
plt.hist(pnl, bins=60, color="#4c78a8", alpha=0.78)
for key, value in risk.items():
    if key.startswith("VaR"):
        plt.axvline(-value, linestyle="--", label=key)
plt.title("Monte Carlo portfolio P&L distribution with VaR markers")
plt.xlabel("Portfolio P&L")
plt.ylabel("Scenario count")
plt.legend()
save_current_figure("05_monte_carlo_pnl_var_es.png")

losses = -pnl
var95 = risk["VaR_95"]
es95 = risk["ES_95"]
plt.figure()
plt.hist(losses, bins=60, color="#72b7b2", alpha=0.78)
plt.axvline(var95, color="black", linestyle="--", label="VaR 95")
plt.axvline(es95, color="#d62728", linestyle="-", label="ES 95")
plt.axvspan(var95, losses.max(), color="#d62728", alpha=0.18, label="Expected Shortfall tail")
plt.title("Loss distribution with VaR and Expected Shortfall region")
plt.xlabel("Portfolio loss")
plt.ylabel("Scenario count")
plt.legend()
save_current_figure("05_loss_distribution_var_es_region.png")
pd.DataFrame([risk])
